<b>Inventory Planning Problem</b>

In [13]:
# Data

# Multiperiod models rely on an ordered set of time periods.
# Since standard sets in Python are _not_ ordered, we will 
# use an integer range, which is ordered (numbers 1 through 4):
Months = range(1,5)
Demand = {1: 100, 2 : 150, 3 : 150, 4 : 250}
ProductionCost = {1 : 11, 2 : 14, 3 : 15, 4 : 16}
ProductionCapacity = 200
HoldingCost = 2
StorageCapacity = 100

In [14]:
# define additional data
ExtraProductionCost = {}
for m in Months:
    ExtraProductionCost[m] = ProductionCost[m] + 0.5

ExtraHoldingCost = 3

In [15]:
ExtraProductionCost

{1: 11.5, 2: 14.5, 3: 15.5, 4: 16.5}

In [16]:
from docplex.mp.model import Model
mdl = Model()

In [17]:
# variables
Produced = mdl.continuous_var_dict(Months, lb=0, ub=ProductionCapacity, name='Production in month')

In [18]:
# new variables
ExtraProduced = mdl.continuous_var_dict(Months, lb=0, name='Extra Production in month')

In [19]:
# variables: inventory at end of month
Inventory = mdl.continuous_var_dict(Months, lb=0, ub=StorageCapacity, name='Inventory at end of month')

In [20]:
# new variables: inventory at end of month
ExtraInventory = mdl.continuous_var_dict(Months, lb=0, name='Extra Inventory at end of month')

In [21]:
# objective: minimize cost
TotalProdCost = mdl.sum(ProductionCost[m]*Produced[m] + ExtraProductionCost[m]*ExtraProduced[m] for m in Months)
TotalHoldingCost = mdl.sum(HoldingCost*Inventory[m] + ExtraHoldingCost*ExtraInventory[m] for m in Months)
mdl.minimize(TotalProdCost + TotalHoldingCost)

mdl.add_kpi(TotalProdCost, 'Total production cost')
mdl.add_kpi(TotalHoldingCost, 'Total holding cost')

DecisionKPI(name=Total holding cost,expr=2Inventory at end of month_1+2Inventory at end of month_2+2Inven..)

In [22]:
# inventory balance constraints
for m in Months:
    if m==1:
        mdl.add_constraint(Inventory[1] + ExtraInventory[1] == Produced[1] + ExtraProduced[1] - Demand[1])
    else:
        mdl.add_constraint(Inventory[m] + ExtraInventory[m] == 
                           ExtraInventory[m-1] + Inventory[m-1] + Produced[m] + ExtraProduced[m] - Demand[m])

In [23]:
# solve 
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00275517,status='optimal')

In [24]:
mdl.print_solution()

objective: 9375.000
status: OPTIMAL_SOLUTION(2)
  "Production in month_1"=200.000
  "Production in month_2"=50.000
  "Production in month_3"=150.000
  "Production in month_4"=200.000
  "Extra Production in month_4"=50.000
  "Inventory at end of month_1"=100.000


In [25]:
mdl.report()

* model docplex_model2 solved with objective = 9375.000
*  KPI: Total production cost = 9175.000
*  KPI: Total holding cost    = 200.000
